<font color=red>**Danger zone:**</font> you'll be fine-tuning a model to generate positive, negative or even toxic reviews. We'll be doing this for fun, but this is also the technique for [review bombing](https://en.wikipedia.org/wiki/Review_bomb), bot farms on social media and other less than dignified stuff. It is ultimately your decision how you apply this knowledge, but before you choose, ask yourself: is this why you chose to learn ML?


# LLMs Alignment with Reinforcement Learning from human feedback (RLHF).

_based on the [original notebook](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) by Ilya Boytsov for the Oxford LLMs workshop_



In this session, you're gonna fine-tune a language model with reinforcement learning to make it generate good (or bad) reviews.

To perform RL-based fine-tuning, we'll use a new (in this course) library called [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). TRL implements the main reinforcement learning components of RLHF: reward modeling and fine-tuning with PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

In [1]:
# %pip install -U -q trl transformers datasets peft
# %pip install -q datasets==4.0.0

### Tutorial: align the model to generate positive movie reviews

To see how TRL works, we'll use it to align GPT2 on IMDB dataset to generate positive (or negative) movie reviews. In fact, __it's your choice whether you want positive or negative reviews.__

But before you choose, let's take a look at the baseline model: a GPT-2 fine-tuned on generating arbitrary movie reviews.

In [2]:
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)

In [3]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated text: The movie was a disappointment, it sucked the life out of it and a lot of the film made the characters disappear...<|endoftext|>


If you run this cell a couple of times, you'll see that the model generates both positive, negative and neutral reviews in some proportion. What we're gonna do next is teach the model to generate more positive (or negative) reviews.

Similarly to InstructGPT, we're gonna do that in 2 stages:
- **train a reward model** to assign higher values to positive (or negative) reviews
- fine-tune the language model to **maximize that reward using [proximal policy optimization](https://openai.com/research/openai-baselines-ppo)**



## Stage 1: train a reward model (1 point)

First, we'll train a BERT-like model as our reward model. We'll generate a synthetic pairwise rankings to emulate human rankings.

__Q:__ why do I need a reward model? Can I just use a pre-trained sentiment classifier? <br> __A:__ Yes, you can - but that only works for movie reviews. But this tutorial will teach you how to do RLHF for any kind objective.


__If you actually want to maximize sentiment (or other "label") instead of human preferences, train reward model as a classifier! (see week5)__


In [4]:
reward_model_name = "distilgpt2"

reward_model = transformers.AutoModelForSequenceClassification.from_pretrained(reward_model_name, device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained(reward_model_name)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at distilgpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# reward_tokenizer.eos_token = reward_tokenizer.sep_token
# reward_tokenizer.pad_token = reward_tokenizer.sep_token

# original_forward = reward_model.forward
# def patched_forward(*args, **kwargs):
#     kwargs.pop("use_cache", None)
#     return original_forward(*args, **kwargs)

# reward_model.forward = patched_forward

__Note that__ the reward model has a separate tokenizer, different from the main model. They don't need to be the same for RLHF fine-tuning.

In [6]:
# To train a reward model, you need a dataset (or generator) of positive-negative pairs.
# Each training sample should be a dict with 4 keys:
#  - input_ids_chosen, attention_mask_chosen = tokenizer("A sentence that human labeler likes more")
#  - input_ids_rejected, attention_mask_rejected = tokenizer("A sentence that human labeler likes less")

import torch
import datasets

class IMDBPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, imdb, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['text'] for row in imdb if row['label'] == accepted_label]
        self.rejected_texts = [row['text'] for row in imdb if row['label'] != accepted_label]
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [7]:
import itertools
import random

def make_iterable_dataset(imdb, tokenizer, accepted_label: int):
    def iterate():
        chosen = [row['text'] for row in imdb if row['label'] == accepted_label]
        rejected = [row['text'] for row in imdb if row['label'] != accepted_label]
        random.shuffle(chosen)
        random.shuffle(rejected)

        for ch, rj in itertools.product(chosen, rejected):
            yield {
                "chosen": ch, "rejected": rj
            }
            # ch_tok = tokenizer(ch, truncation=True)
            # rj_tok = tokenizer(rj, truncation=True)
            # yield dict(
            #     input_ids_chosen=ch_tok['input_ids'],
            #     attention_mask_chosen=ch_tok['attention_mask'],
            #     input_ids_rejected=rj_tok['input_ids'],
            #     attention_mask_rejected=rj_tok['attention_mask']
            # )
    return datasets.IterableDataset.from_generator(iterate)

In [8]:
TARGET_LABEL = 0   # and make sure it works by reviewing the sample printed below
imdb = datasets.load_dataset("imdb", split='train')
reward_data_old = IMDBPairwiseDataset(imdb, reward_tokenizer, accepted_label=TARGET_LABEL)
reward_data = make_iterable_dataset(imdb, reward_tokenizer, accepted_label=TARGET_LABEL)


# OLD API
# sample = reward_data[31337]
# print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
# print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

sample = next(iter(reward_data))
print('CHOSEN:', sample['chosen'])
print('REJECTED:', sample['rejected'])

Found 12500 chosen and 12500 rejected texts, 156250000 pairs
CHOSEN: A slasher flick, made in the early 80's, has a curse on it which has anyone who tries to finish it turning up dead. Years later, a group of film students attempted to complete the movie - also resurrecting the films deadly curse. Great idea for a film, but sadly 'Cut' is just another wasted opportunity.<br /><br />Unfortunately Australia hasn't had the world's best track record when it comes to horror. 'Razorback' (1984) was an out and out dud as was 'Holwing III' (1987), which was half an American film anyway. As for our foray into comedy-horror, 'Body Melt' (1993) is best left forgotten. The problem with 'Cut' is that the makers trying to create a clever horror satire a la 'Scream' (1996) but have no insight into the genre or what makes it work. And although this sounds weird me saying this about a slasher film but what 'Cut' really lacks is any "heart". Sure it follows the basic "rules" established by 'Scream', but

We'll be using `trl.RewardTrainer` - a special case of `transformers.Trainer` that you used in the past. `RewardTrainer` accepts the same format of training arguments (e.g. batch size, gradient checkpointing) as before, except that it trains the model for the pairwise reward objective from [the InstructGPT paper](https://arxiv.org/pdf/2203.02155.pdf):

![img](https://i.imgur.com/2JzNAPs.png)

Note that the model itself does not score pairs: it processes chosen ($y_w$) and rejected ($y_l$) samples independently. To minimize this loss, the reward model needs to score chosen sample higher than the rejected one. Note that the formula also assumes some context $x$, which is useful for seq2seq tasks. In our case of movie reviews, $x$ is empty.

In [9]:
import trl
import peft

peft_config = peft.LoraConfig(task_type=peft.TaskType.SEQ_CLS, r=16, lora_alpha=16, target_modules=["c_proj", "c_attn"])

training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=700,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,                    # disable this on CPU or on very old GPUs
    # you may add any other hyperparameters that you found useful in weeks 5-7
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    processing_class=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=peft_config,  # optionally, you may tune with LoRA, prompt-tuning, etc
)

trainer.train()

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
/home/vagiz/Desktop/desktop_vagiz/yandex/nlp_course/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (1073 > 1024). Running this sequence through the model will result in indexing errors


Step,Training Loss
50,0.471500
100,0.398700
150,0.317500
200,0.216700
250,0.140200
300,0.076500
350,0.030400
400,0.012900
450,0.003700
500,0.001800


TrainOutput(global_step=700, training_loss=0.11952506579458713, metrics={'train_runtime': 134.5402, 'train_samples_per_second': 20.812, 'train_steps_per_second': 5.203, 'total_flos': 750751965315072.0, 'train_loss': 0.11952506579458713, 'epoch': 1.0})

In [10]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): lora.Linear(
            (base_layer): Conv1D(nf=2304, nx=768)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=768, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=2304, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (c_proj): lora.Linear(
            (base_layer): Conv1D(nf=768, nx=768)
            (lora_dr

### Sanity-check the reward model (1 point)

Let's check how our reward model performs.

__Your task__ is to measure how often does your reward model can rank a pair of (chosen and rejected) reviews correctly. Please measure this separately for train data (`imdb`) and a separate test set loaded below.

In [11]:
for sample_index in 45, 16000:
  print('TEXT:', imdb[sample_index]['text'])
  inputs = reward_tokenizer(
      imdb[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', imdb[sample_index]['label'])
  print()

# note: your reward model may produce different absolute rewards.
# This is fine as long as the rewards are ordered correctly (most of the time)

TEXT: This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get "terrorized" by this pathetic "crazed killer", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.
REWARD: 3.5319876670837402
LABEL: 0

TEXT: Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes 

In [12]:
import numpy as np
from tqdm.notebook import tqdm

imdb_test = datasets.load_dataset("imdb", split='test')
reward_data_test = make_iterable_dataset(imdb_test, reward_tokenizer, accepted_label=TARGET_LABEL)

def get_accuracy(data, max_num_samples: int = 100, num_update: int = 100):
    num_samples = 0
    num_correct = 0

    data_iter = iter(reward_data)
    pbar = tqdm(total=max_num_samples, desc="Evaluating accuracy")
    while num_samples < max_num_samples:
        sample = next(data_iter)
        chosen = reward_tokenizer(sample["chosen"], truncation=True, return_tensors='pt').to(device)
        rejected = reward_tokenizer(sample["rejected"], truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            reward_chosen = reward_model(**chosen).logits[0, 0].item()
            reward_rejected = reward_model(**rejected).logits[0, 0].item()
        num_samples += 1
        num_correct += reward_chosen > reward_rejected

        pbar.update(1)
        if num_samples % num_update == 0:
            pbar.set_postfix({"acc": num_correct / num_samples})
    return num_correct / num_samples

In [13]:
accuracy_train = get_accuracy(reward_data, max_num_samples=10_000)
accuracy_test = get_accuracy(reward_data_test, max_num_samples=10_000)

Evaluating accuracy:   0%|          | 0/10000 [00:00<?, ?it/s]

Evaluating accuracy:   0%|          | 0/10000 [00:00<?, ?it/s]

In [14]:
print(f"Accuracy (train): {accuracy_train:.2f}")
print(f"Accuracy (test): {accuracy_test:.2f}")

Accuracy (train): 0.74
Accuracy (test): 0.98


### Reward-guided generation (1 point)

If you did everything right, by now you should have a decent reward model. Before we use it for reinforcement learning, let's see if we can align model samples without any training.

To do so, you can use reward-guided inference: __generate N=16 samples, then select the one with the highest reward__ (according to your reward model).

For this problem, it's on you to demonstrate whether or not your code works. Find at least 5 neutral prompts such as "This movie is" (...), generate samples, rank them based on reward and show which samples get the highest reward.

Note: it is faster to generate samples in parallel, rather than sequentially, as follows:




In [15]:
inputs = main_tokenizer(["It was"] * 5, return_tensors='pt').to(device)
for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
    print("Sample:", main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: It was too long and the story was hard to follow at times. (The fact that there was a script for this movie is a mystery to watch out for.) So I had to do a little research in the hope that some of the references may not have
Sample: It was the real me, too. I was very upset when I realized my boyfriend was the other reason that his body was never found. I got up, walked out the next morning, and then just watched him walk his dog to the toilet. To make
Sample: It was a beautiful day and I'm glad I came back. I had seen the first trailer in a while and this one was a complete mess. Too bad, a lot of people are afraid to take this movie seriously.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was never a big issue that has been solved for me, but that is a really big reason why this film was very enjoyable.<br /><br />This is about a kid who loses a lot. He also loses a very old friend in a way
Sample: It was a movie you sa

In [16]:
N = 16
neutral_prompts = ["My opinion is", "The movie is", "This is the", "To be honest,", "It was a rare moment"]

for prompt in neutral_prompts:
    inputs_main = main_tokenizer([prompt] * N, return_tensors='pt').to(device)
    inputs_reward = reward_tokenizer([prompt] * N, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = main_model.generate(**inputs_main, max_new_tokens=50, do_sample=True)
        rewards = reward_model(outputs).logits[:, 0].cpu().flatten()
    best_index = torch.argmax(rewards)
    print("Sample:", main_tokenizer.decode(outputs[best_index].flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: My opinion is that the movie is so poorly done that it should have no place on the list of movies that "just doesn't work". This is one of the problems with an indie, if you want real movies you must have some real "real stuff". My


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: The movie is a lot more entertaining and interesting than your average college romance flick. In fact the movie was so funny at times I knew what to expect. One of the main characters is supposed to be a "manhunter" for evil and has his own troubles.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: This is the second time I've seen this film, but I've seen it twice. It's brilliant. You've got to make sure you put your mind over it. Also, there are some really big fights, especially with the demon (and you'll be


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: To be honest, I can't see why anyone would bother renting this film - it's too silly.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was a rare moment in any movie for American actors to look at a screen and say "I think I've found the right actor. Let's have fun," and I think Mr. Osmond was right.<br /><br />One of the major issues I


We see many examples are negative.

# Stage 2: fine-tune the main model with RL (2 points)


For this tutorial, we will optimize GPT2 to produce positive IMDB movie reviews using the reward model you trained above.

Unlike supervised fine-tuning, RL allows model to generate it's own sentences on each training step. Then, it calculates the reward of those specific sentences, and finally, updates the model to increase the probability of sentences with high reward.

Thus, each RLHF consists of three stages: __Rollout__, __Evaluation__ and __Update__

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

The update stage depends on the specific RL algorithm. We'll be using Proximal Policy Optimization, or [PPO](https://arxiv.org/abs/1707.06347), similarly to what was used for InstructGPT.

Before we run those 3 stages, however, we need to create a dataset of "queries" - partial reviews in our case.

In [17]:
# Note: this code is specific to IMDB; you will need to re-write it for other tasks
imdb_for_rlhf = imdb.filter(lambda row: len(row['text']) > 200, batched=False)
imdb_for_rlhf = imdb_for_rlhf.remove_columns(['label'])
def length_sampler(min_val=2, max_val=8):
    return random.randint(min_val, max_val)
sample_length = length_sampler
# sample_length = trl.core.LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"], return_tensors='pt')[0][: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

imdb_for_rlhf = imdb_for_rlhf.map(select_query_and_tokenize, batched=False)
imdb_for_rlhf.set_format(type="torch")

In [18]:
imdb_for_rlhf[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Next, let's prepare your reward model to predict rewards on whatever reviews were generated. Note that we use plaintext reviews because main model uses a different tokenizer from the reward model.

In [19]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
    inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        return reward_model(**inputs).logits[:, 0]

In [20]:
compute_reward([imdb[45]['text'], imdb[16000]['text']])  # test on human-written reviews

tensor([3.5320, 1.9213], device='cuda:0')

Finally, we move to RL training. In this tutorial, we'll train LoRA adapters and not the full model.

In [21]:
import peft
from torch import nn

peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9391


/home/vagiz/Desktop/desktop_vagiz/yandex/nlp_course/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Same as before, trl has a special type of trainer that minimize PPO-specific pseudo-loss. You can read more on this trainer [here](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [22]:
ref_model = transformers.AutoModelForCausalLM.from_pretrained(reward_model_name)

value_model = transformers.AutoModelForCausalLM.from_pretrained(reward_model_name)
value_model.score = nn.Linear(value_model.config.n_embd, 1, bias=False)
nn.init.normal_(value_model.score.weight, mean=0.0, std=0.02)

main_model.model.generation_config = transformers.GenerationConfig(
    max_new_tokens=128,
    do_sample=True,
    top_p=0.95,
    pad_token_id=main_tokenizer.pad_token_id,
    eos_token_id=main_tokenizer.eos_token_id,
)
main_model.model.is_gradient_checkpointing = True

In [23]:
training_args = trl.PPOConfig(
    # model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    batch_size=64,
    num_ppo_epochs=4,                 # PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    args=training_args,
    model=main_model.model,
    processing_class=main_tokenizer,
    train_dataset=imdb_for_rlhf,
    data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0]),
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

ppo_trainer.use_cpu_amp = False
ppo_trainer.label_smoother = None
ppo_trainer.compute_loss_func = None
ppo_trainer.model_accepts_loss_kwargs = True
# ppo_trainer.train()

/home/vagiz/Desktop/desktop_vagiz/yandex/nlp_course/.venv/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:200: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


Below, I got a problem that was not able to fix. There were too many inconsistencies between the code in new versions and code in later versions. Even though I was able to rewrite previous code to run in latest versions, here, I stopped.

I also found this PR (https://github.com/huggingface/trl/pull/3410) about introducing `PPOTrainer.step()` again, as it was removed in latest versions. I tried to use `PPOTrainer.training_step()`, but it did not work.

In [24]:
from tqdm.auto import tqdm
max_steps = 50   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=128, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id)
#                                  ^-- task-specific parameter!

ppo_trainer.optimizer.zero_grad()
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
    for epoch, batch in progressbar:
        if epoch >= max_steps:
            break

        # Rollout stage: generate continuations from batch queries using main_model
        batch = main_tokenizer(batch['text'], truncation=True, return_tensors='pt', padding=True, padding_side='left').to(device)
        response_tensors = main_model.model.generate(batch['input_ids'], **generation_kwargs)
        # ^-- list of tensors of token ids from main model tokenizer

        # de-tokenize responses to strings (since reward model uses a different tokenizer)
        batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
        # note: response_tensors already contain query tokens, so we don't need to add queries manually.
        # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]

        # Evaluation stage
        rewards = compute_reward(batch['response'])

        # Update stage
        print(batch["input_ids"].shape)
        print(response_tensors.shape)
        print(rewards.shape)
        stats = ppo_trainer.training_step(
            ppo_trainer.model,
            dict(
                input_ids=batch["input_ids"],
                response_tensors=response_tensors,
                rewards=rewards
            )
        )
        stats['rewards/mean'] = rewards.mean().item()
        
        print("-" * 30, 'STEP', epoch, '-' * 30)
        print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
        print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
        print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
        print()

        ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  0%|          | 0/50 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


torch.Size([8, 622])
torch.Size([8, 750])
torch.Size([8])


TypeError: 'NoneType' object is not subscriptable

# [Optional] high-effort bonus assignment: RL fine-tuning in the wild


Your main task for this week is to use the RLHF pipeline to train a model for a reward of your choice. Here's what you can choose from:

__A. Toxicity fine-tuning:__ train the model to be less (or more!) toxic. For this task, you may use the data from [jigsaw toxic comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) and [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat),  or any other source. Alternatively, you may use toxicity scores from [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).


__B. Actual human feedback:__ use one of the existing datasets with pairwise human feedback to align your langauge model. You may use [anthropic's hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf), [OpenAssistant dataset](https://huggingface.co/datasets/OpenAssistant/oasst1) or any other data you see fit. You may also turn the tables and train the model to [minimize](https://habrastorage.org/getpro/geektimes/post_images/ac7/2ad/827/ac72ad82767d4132164a4b6b76196c42.jpg) human preferences, as long as your model does not degrade to gibberish.

__C. Controlled generation:__ Instead of training a reward model from human feedback, you may define the reward function as the text length (longer or shorter) or number of times the model uses specific words (e.g. "sorry", "apologize"). If you choose specific words, make sure the model generates them at least sometimes.

__Alternatively,__ you may choose a different task. However, unless your task is very similar to one of the above, there is a chance that it will be **significantly** harder to solve, requiring orders of magnitude more compute and tuning. If you are in doubt, please ask the course staff. If they are AFK (again >.<), please prefer one of the recommended tasks.


#### General tips & tricks


Things to look out for:
- during PPO stage, the reward model should be in eval mode (dropout disabled)
- make sure max_length and max_new_tokens are enough for your chosen dataset - at least most of the time
- when in doubt, view the data manually or inspect how the model performs on a few samples


We highly recommend that you manually check the performance after each sub-stage:
1. when you assembled the pairwise dataset, inspect a couple of from of *your* dataset class and detokenize them. Make sure that you-the-human understand why one sample was accepted and the other - rejected. At least most of the time. This also lets you spot tokenization/truncation errors.
2. after you trained a reward model, measure how accurate this model is in isolation. If your reward model is poor, any subsequent RLHF will also fail.
3. once you've trained the main model with RL, ask it to generate examples and explore how well it does. If it produces an obviously bad output, check if the reward model assigns high reward to that output. If yes, reward model is the culprit; if no, it's a question of better/longer PPO training.

__It is also a good idea to periodically print samples during training.__

__When stuck, simplify the problem.__ If you've spent a several hours enchanting the reward model but it still won't budge, try switching to a simple subtask. For instance, if you're training on hh-rlhf, try limiting it the dataset to 10% of the shortest sequences - they are typically easier to learn.


## Bonus Assignment Stages

Regardless of the specific task you chose, your solution needs to contain several parts that will be graded separately (for bonus points).


#### Stage 1: reward model

Construct a dataset for training the reward model on your problem. Then, train a reward model on that dataset and evaluate how well can your model predict preferences on a hold-out (test) subset of your data.

Please make sure that the part of your notebook where you evaluate reward model is clearly visible and reasonably easy to read. And for all that is holy, do not call it IMDB unless it actually **is** data of imdb movie reviews :)

__Not all tasks require a reward model for later PPO fine-tuning.__ For instance, there's no reason to train a reward model if your reward equals sentence length. Likewise, toxicity reward can be estimated with a pre-trained toxicity classifier. __If your task does not require training a reward model, please train an unrelated model on [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) as though you were solving assignment version B.__ This is for grading purposes only, you won't use this model for stage 2.


#### Stage 2: RL fine-tuning

Once the reward model is ready - or you can compute rewards without a model - it is time to maximize that reward with PPO. Optionally, you may replace PPO with another RL algorithm (or unlikelihood learning scheme), but only if you're feeling adventurous.


First, you need to choose a language model to be fine-tuned. You may choose any model, but make sure that your model **can** generate the data in your format. For instance, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) is a general purpose LM and may (or may not) need prompt engineering to generate chat assistant responses. For that reason, it is best if you **do not use `"lvwerra/gpt2-imdb"` unless you're generating only movie reviews**.



There are two "difficulty modes" for this task:
For the **easy mode**, use [gpt2-large](https://huggingface.co/gpt2-large) or [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) with minimal code changes.
If you want the **Hard mode:** use a larger (e.g. 7B) model in combination with `load_in_4bit` and LoRA, the same way we did last week.
Some reasonable model choices are [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7b](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) for general-purpose LM or [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b), [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) for chat-based tasks, though there are many more (see [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)). In the hard mode, you will need to modify the training arguments to enable 4-bit fine-tuning. Furthermore, your experiments will take somewhat longer to complete. On the plus side, your model will produce significantly better results.

__High reward is not enough!__ RL algorithms are famous for [cheating their reward functions](https://openai.com/research/faulty-reward-functions). To ensure that your model is actually doing what you want it to do, you will need some additional evaluation. To get the full grade, provide at least 20 side-by-side examples of your fine-tuned model vs original model predictions and a short summary.

Alternatively, you may provide 5 examples and some extrinsic evaluation metric over many examples. For instance, you may use a different pre-trained toxicity score for option A. When dealing with human preferences, you may choose to [enlist actual humans](https://toloka.ai/) or [ask GPT/Claude](https://arxiv.org/pdf/2304.03277.pdf) to compare your model's predictions. For task C, when optimizing for simple rewards like sentence lengths, it is enough to compare histograms of rewards (e.g. average lengths).










